## Setup — Define Dataset Paths

In [ ]:
from pathlib import Path
repo_root = Path(r"D:/ML_CP")
clean_path = repo_root / "data" / "processed" / "flowcred_cleaned.csv"
raw_path = repo_root / "data" / "raw" / "dataset.csv"
validated_output = repo_root / "data" / "processed" / "flowcred_validated.csv"
print("repo_root:", repo_root)
print("clean_path:", clean_path)
print("raw_path:", raw_path)
print("validated_output:", validated_output)


## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Libraries imported successfully.")


## Step 2 — Load Cleaned Dataset

In [ ]:
clean_path = Path(r"D:/ML_CP/data/processed/flowcred_cleaned.csv")
df = pd.read_csv(clean_path)

print("Shape:", df.shape)
print("\nFirst 10 rows:")
display(df.head(10))
print("\nLast 10 rows:")
display(df.tail(10))
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)


## Step 2A — Parse Date Columns for Validation

In [ ]:
for col in ["invoice_date", "due_date", "payment_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Updated date columns to datetime64[ns]:")
print([col for col in ["invoice_date", "due_date", "payment_date"] if col in df.columns])
print("\nUpdated dtypes:")
display(df[[col for col in ["invoice_date", "due_date", "payment_date"] if col in df.columns]].dtypes)


## Step 3 — Verify Expected Columns

In [ ]:
expected_columns = [
    "invoice_id",
    "customer_id",
    "invoice_amount",
    "invoice_date",
    "due_date",
    "payment_date",
    "delay_days",
    "is_late",
    "invoice_status",
]

present_columns = [col for col in expected_columns if col in df.columns]
missing_columns = [col for col in expected_columns if col not in df.columns]
unexpected_columns = [col for col in df.columns if col not in expected_columns]

print("Present columns:")
print(present_columns)
print("\nMissing expected columns:")
print(missing_columns)
print("\nUnexpected additional columns:")
print(unexpected_columns)

if missing_columns:
    print("\nWarning: required validation columns are missing.")
else:
    print("\nAll required validation columns are present.")


## Step 4 — Investigate Missing Invoice IDs

In [ ]:
missing_invoice_ids = df["invoice_id"].isna().sum()
missing_pct = (missing_invoice_ids / len(df)) * 100 if len(df) else 0
missing_rows = df[df["invoice_id"].isna()].copy()

print("Number of missing invoice_id values:", missing_invoice_ids)
print("Percentage missing:", round(missing_pct, 2), "%")
print("\nRows where invoice_id is missing:")
display(missing_rows)

valid_quality_mask = (
    df["invoice_id"].isna() &
    df["customer_id"].notna() &
    df["invoice_amount"].notna() &
    df["invoice_date"].notna()
)
print("\nRows with otherwise valid customer/date/amount information:", int(valid_quality_mask.sum()))
if valid_quality_mask.any():
    display(df.loc[valid_quality_mask, [
        "customer_id", "invoice_amount", "invoice_date", "due_date", "payment_date", "invoice_status", "delay_days", "is_late"
    ]].head(10))

print("\nFinal result: missing invoice_id count =", missing_invoice_ids)


## Step 5 — Investigate Duplicate Invoice IDs

In [ ]:
duplicate_rows_mask = df["invoice_id"].duplicated(keep=False)
rows_in_duplicate_invoice_ids = int(duplicate_rows_mask.sum())
unique_invoice_ids = df["invoice_id"].nunique(dropna=True)
duplicate_examples = df.loc[duplicate_rows_mask, [
    "invoice_id", "customer_id", "invoice_amount", "invoice_date", "due_date", "payment_date", "delay_days", "is_late", "invoice_status"
]].sort_values("invoice_id")

print("Number of unique invoice IDs:", unique_invoice_ids)
print("Number of rows involved in duplicate invoice IDs:", rows_in_duplicate_invoice_ids)
print("\nExamples of duplicate invoice IDs:")
display(duplicate_examples.head(20))

print("\nInterpretation: duplicate invoice IDs are not removed automatically; they are checked as exact duplicates vs same ID with different information.")


## Step 6 — Investigate area_business

In [ ]:
if "area_business" in df.columns:
    area_missing = df["area_business"].isna().sum()
    area_total = len(df)
    area_missing_pct = (area_missing / area_total) * 100 if area_total else 0
    area_unique_values = df["area_business"].nunique(dropna=True)
    print("area_business exists:", True)
    print("Number of non-null values:", area_total - area_missing)
    print("Number of missing values:", area_missing)
    print("Missing percentage:", round(area_missing_pct, 2), "%")
    print("Number of unique values:", area_unique_values)
    print("\nSample unique values:")
    print(df["area_business"].dropna().unique()[:10])
    if area_missing == area_total:
        print("\nRecommendation: area_business can be removed because it contains no information.")
        before_cols = len(df.columns)
        df = df.drop(columns=["area_business"]).copy()
        print("Before columns:", before_cols)
        print("After columns:", len(df.columns))
    else:
        print("\narea_business is not fully empty; it is left in place for now.")
else:
    print("area_business does not exist in the cleaned dataset.")


## Step 7 — Verify the 50,000 → 48,839 Row Difference

In [ ]:
if raw_path.exists():
    raw_df = pd.read_csv(raw_path)
    original_row_count = 50000
    current_row_count = len(df)
    rows_removed = original_row_count - current_row_count
    removal_percentage = (rows_removed / original_row_count) * 100 if original_row_count else 0
    raw_exact_dupes = raw_df.duplicated().sum()
    print("Original rows:", original_row_count)
    print("Current rows:", current_row_count)
    print("Number removed:", rows_removed)
    print("Removal percentage:", round(removal_percentage, 2), "%")
    print("\nRaw exact duplicate rows:", raw_exact_dupes)
    print("Raw duplicate invoice IDs (non-null IDs):", raw_df["invoice_id"].duplicated(keep=False).sum())
    print("Raw missing invoice_id values:", raw_df["invoice_id"].isna().sum())
    print("\nInterpretation: the 1,161-row difference aligns with exact duplicate removal performed in preprocessing.")
else:
    print("Raw dataset is not available in the expected location.")


## Step 8 — Validate Invoice Amounts

In [ ]:
missing_amount = df["invoice_amount"].isna().sum()
zero_amount = (df["invoice_amount"] == 0).sum()
negative_amount = (df["invoice_amount"] < 0).sum()
min_amount = df["invoice_amount"].min()
max_amount = df["invoice_amount"].max()
median_amount = df["invoice_amount"].median()
mean_amount = df["invoice_amount"].mean()

print("Missing invoice_amount:", missing_amount)
print("Zero invoice_amount:", zero_amount)
print("Negative invoice_amount:", negative_amount)
print("Minimum amount:", min_amount)
print("Maximum amount:", max_amount)
print("Median amount:", median_amount)
print("Mean amount:", mean_amount)

large_threshold = df["invoice_amount"].quantile(0.99)
print("\n99th percentile threshold:", large_threshold)
print("Potentially large values above the 99th percentile:")
display(df.loc[df["invoice_amount"] > large_threshold, [
    "invoice_id", "customer_id", "invoice_amount", "invoice_date", "due_date", "payment_date", "invoice_status"
]].head(20))

print("\nInterpretation: missing, zero, and negative values remain visible; they are not silently removed.")


## Step 9 — Validate Customer IDs

In [ ]:
missing_customer = df["customer_id"].isna().sum()
unique_customers = df["customer_id"].nunique(dropna=True)
customer_counts = df["customer_id"].value_counts(dropna=False)

print("Missing customer_id:", missing_customer)
print("Unique customer count:", unique_customers)
print("\nTop 10 customers by invoice count:")
display(customer_counts.head(10).rename_axis("customer_id").reset_index(name="invoice_count"))

print("\nCustomers with only one invoice:", int((customer_counts == 1).sum()))
print("We do not remove customers based solely on invoice frequency.")


## Step 10 — Validate Date Columns

In [ ]:
for col in ["invoice_date", "due_date", "payment_date"]:
    print(f"\n{col}")
    print("Data type:", df[col].dtype)
    print("Missing count:", df[col].isna().sum())
    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())

invoice_gt_due = int((df["invoice_date"] > df["due_date"]).sum())
payment_before_invoice = int((df["payment_date"] < df["invoice_date"]).sum())
payment_before_due = int((df["payment_date"] < df["due_date"]).sum())

print("\ninvoice_date > due_date count:", invoice_gt_due)
print("payment_date < invoice_date count:", payment_before_invoice)
print("payment_date < due_date count:", payment_before_due)

print("\nExamples of invoice_date > due_date:")
display(df.loc[df["invoice_date"] > df["due_date"], ["invoice_id", "invoice_date", "due_date", "payment_date", "invoice_status"]].head(10))
print("\nExamples of payment_date < invoice_date:")
display(df.loc[df["payment_date"] < df["invoice_date"], ["invoice_id", "invoice_date", "due_date", "payment_date", "invoice_status"]].head(10))
print("\nExamples of payment_date < due_date (early payment):")
display(df.loc[df["payment_date"] < df["due_date"], ["invoice_id", "invoice_date", "due_date", "payment_date", "invoice_status"]].head(10))

print("\nInterpretation: early payment is valid; payment_date < invoice_date is flagged for investigation rather than deleted automatically.")


## Step 11 — Validate Delay Days

In [ ]:
calculated_delay_days = (df["payment_date"] - df["due_date"]).dt.days
mismatch_mask = df["delay_days"].notna() & (df["delay_days"] != calculated_delay_days)
print("Number of delay_days mismatches:", int(mismatch_mask.sum()))
if mismatch_mask.any():
    example_df = df.loc[mismatch_mask, ["invoice_id", "due_date", "payment_date", "delay_days"]].copy()
    example_df["calculated_delay_days"] = calculated_delay_days.loc[mismatch_mask].values
    display(example_df.head(20))
else:
    print("No mismatch found between existing delay_days and the recalculated value for rows with payment_date available.")


## Step 12 — Validate is_late

In [ ]:
paid_mask = df["payment_date"].notna()
expected_is_late = pd.Series(np.nan, index=df.index, dtype=float)
expected_is_late.loc[paid_mask] = (df.loc[paid_mask, "delay_days"] > 0).astype(int)
paid_mismatch = paid_mask & df["is_late"].notna() & (df["is_late"].astype(float) != expected_is_late)
open_late_mask = df["invoice_status"].eq("open") & df["is_late"].notna()

print("Paid invoice mismatches between is_late and delay_days > 0:", int(paid_mismatch.sum()))
print("Open invoices incorrectly labeled as late:", int(open_late_mask.sum()))

if paid_mismatch.any():
    display(df.loc[paid_mismatch, ["invoice_id", "due_date", "payment_date", "delay_days", "is_late", "invoice_status"]].head(20))
else:
    print("No mismatches found for paid invoices.")

if open_late_mask.any():
    display(df.loc[open_late_mask, ["invoice_id", "due_date", "payment_date", "delay_days", "is_late", "invoice_status"]].head(20))
else:
    print("No open invoices were incorrectly labeled as late.")

print("\nImportant: open invoices must not be labeled as late.")


## Step 13 — Validate invoice_status

In [ ]:
df["invoice_status"] = df["invoice_status"].astype(str).str.lower()
expected_status = np.where(df["payment_date"].notna(), "paid", "open")
status_mismatch = df["invoice_status"] != expected_status

print("Paid count:", (df["invoice_status"] == "paid").sum())
print("Open count:", (df["invoice_status"] == "open").sum())
print("Status mismatches:", int(status_mismatch.sum()))
if status_mismatch.any():
    display(df.loc[status_mismatch, ["invoice_id", "payment_date", "invoice_status"]].head(20))
else:
    print("No invoice_status mismatches found.")

print("\nExpected logic: payment_date.notna() -> paid; payment_date.isna() -> open")


## Step 14 — Check Leakage-Prone Columns

In [ ]:
leakage_prone = ["payment_date", "delay_days", "is_late", "invoice_status"]
print("Leakage-prone outcome columns:")
for col in leakage_prone:
    print("-", col)

print("\nThese columns must NOT be used as prediction features for pre-payment prediction.")
print("\nSafe raw input candidates:")
raw_input_candidates = [
    col for col in df.columns
    if col not in leakage_prone and col not in ["invoice_id", "customer_id"]
]
print(raw_input_candidates[:20])
print("\nTemporal or derived features require leakage-safe engineering later.")


## Step 15 — Final Data Quality Report

In [ ]:
is_late_mismatch_mask = (df["payment_date"].notna()) & (df["is_late"].astype(float) != ((df["delay_days"] > 0).astype(float)))
status_mismatch_mask = df["invoice_status"].astype(str).str.lower() != np.where(df["payment_date"].notna(), "paid", "open")
report = pd.DataFrame([
    ["Total rows", len(df)],
    ["Total columns", len(df.columns)],
    ["Missing invoice IDs", df["invoice_id"].isna().sum()],
    ["Duplicate invoice IDs", int(df["invoice_id"].duplicated(keep=False).sum())],
    ["Exact duplicate rows", int(df.duplicated().sum())],
    ["Missing customer IDs", df["customer_id"].isna().sum()],
    ["Missing invoice amounts", df["invoice_amount"].isna().sum()],
    ["Zero invoice amounts", int((df["invoice_amount"] == 0).sum())],
    ["Negative invoice amounts", int((df["invoice_amount"] < 0).sum())],
    ["Invalid date relationships", int(((df["invoice_date"] > df["due_date"]) | (df["payment_date"] < df["invoice_date"]) | (df["payment_date"] < df["due_date"])).sum())],
    ["Delay calculation mismatches", int(((df["delay_days"].notna()) & (df["delay_days"] != (df["payment_date"] - df["due_date"]).dt.days)).sum())],
    ["is_late mismatches", int(is_late_mismatch_mask.sum())],
    ["invoice_status mismatches", int(status_mismatch_mask.sum())],
    ["Empty columns", int(sum(df[col].isna().all() for col in df.columns))],
    ["Rows removed from original dataset", 1161],
], columns=["Metric", "Result"])

display(report)


## Step 16 — Decide Required Cleanup

In [ ]:
if "area_business" in df.columns and df["area_business"].isna().all():
    validated_df = df.drop(columns=["area_business"]).copy()
    print("Action: removed completely empty column area_business")
    print("Rows affected:", 0)
    print("Columns affected:", 1)
    print("Reason: area_business is 100% missing and contains no usable information.")
    print("Before column count:", len(df.columns))
    print("After column count:", len(validated_df.columns))
else:
    validated_df = df.copy()
    print("No automatic cleanup required based on validation evidence.")
    print("Rows affected:", 0)
    print("Columns affected:", 0)


## Step 17 — Create Final Validated Dataset

In [ ]:
validated_output = Path(r"D:/ML_CP/data/processed/flowcred_validated.csv")
validated_df.to_csv(validated_output, index=False)
print("Saved final validated dataset:", validated_output)
print("File exists:", validated_output.exists())
print("\nFinal shape:", validated_df.shape)
print("Final columns:")
print(validated_df.columns.tolist())
print("\nFinal missing-value summary:")
display(validated_df.isna().sum().sort_values(ascending=False).reset_index())


## Step 18 — Final Validation Assertions

In [ ]:
required_cols = [
    "invoice_id",
    "customer_id",
    "invoice_amount",
    "invoice_date",
    "due_date",
    "payment_date",
    "delay_days",
    "is_late",
    "invoice_status",
]

assert all(col in validated_df.columns for col in required_cols), "Required columns are missing."
paid_mask = validated_df["payment_date"].notna()
if paid_mask.any():
    assert (validated_df.loc[paid_mask, "delay_days"] == (validated_df.loc[paid_mask, "payment_date"] - validated_df.loc[paid_mask, "due_date"]).dt.days).all(), "delay_days mismatch on paid invoices."
    assert (validated_df.loc[paid_mask, "is_late"].astype(float) == ((validated_df.loc[paid_mask, "delay_days"] > 0).astype(float))).all(), "is_late mismatch on paid invoices."
open_mask = validated_df["invoice_status"].eq("open")
assert validated_df.loc[open_mask, "payment_date"].isna().all(), "Open invoices must have missing payment_date."
assert validated_df.loc[open_mask, "delay_days"].isna().all(), "Open invoices must have missing delay_days."
assert validated_df.loc[open_mask, "is_late"].isna().all(), "Open invoices must have missing is_late."
assert (validated_df["invoice_status"].astype(str).str.lower() == np.where(validated_df["payment_date"].notna(), "paid", "open")).all(), "invoice_status does not match payment_date availability."
print("All validation assertions passed.")


## Step 19 — Final Notebook Summary

In [ ]:
target_mismatch_mask = (validated_df["payment_date"].notna()) & (validated_df["is_late"].astype(float) != ((validated_df["delay_days"] > 0).astype(float)))
print("Data-quality validation completed.")
print("\nSummary:")
print("Total validated rows:", len(validated_df))
print("Total columns:", len(validated_df.columns))
print("Missing invoice IDs:", validated_df["invoice_id"].isna().sum())
print("Duplicate invoice IDs:", int(validated_df["invoice_id"].duplicated(keep=False).sum()))
print("Empty columns:", int(sum(validated_df[col].isna().all() for col in validated_df.columns)))
print("Invalid date records:", int(((validated_df["invoice_date"] > validated_df["due_date"]) | (validated_df["payment_date"] < validated_df["invoice_date"]) | (validated_df["payment_date"] < validated_df["due_date"])).sum()))
print("Delay calculation mismatches:", int(((validated_df["delay_days"].notna()) & (validated_df["delay_days"] != (validated_df["payment_date"] - validated_df["due_date"]).dt.days)).sum()))
print("Target mismatches:", int(target_mismatch_mask.sum()))
print("Rows removed during validation:", 0)
print("Output path:", validated_output)
print("\nThe dataset is now ready for the next stage: leakage-safe feature engineering.")
